In [4]:
import gradio as gr

def greet(name):
    return "你好 " + name

gr.Interface(fn=greet, inputs="text", outputs="text").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [5]:
import gradio as gr 

def greet(name):
	return "你好: " + name + "!"

inputs = gr.Textbox(lines=2, placeholder="請輸入姓名...", label="請輸入使用者姓名") 
outputs = gr.Label() 
examples = ["陳會安", "江小魚"] 
app = gr.Interface(fn=greet, 
                   inputs=inputs, 
                   outputs=outputs, 
                   examples=examples, 
                   title = "歡迎使用者", 
                   description = "輸入姓名顯示歡迎訊息") 
app.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [6]:
def greet(name, is_lady, fahrenheit): 	
	if is_lady: greeting = name + "女士你好!" 
	else: greeting = name + "男士你好!"  
	greeting += "今天溫度是華氏: " + str(fahrenheit) 	
	celsius = (fahrenheit - 32) * 5 / 9 	
	return greeting, round(celsius, 2) 

app = gr.Interface(fn=greet, 
                   inputs=["text", "checkbox", gr.Slider(0, 100)],
                   outputs=["text", "number"]) 
app.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [7]:
import numpy as np 
from PIL import Image 
import gradio as gr 
def rgb2gray(input): 	
	img = Image.fromarray(input)
	img = img.convert('L') 
	return np.array(img)
app = gr.Interface(rgb2gray, 
                   gr.Image(image_mode="RGB"), 
                   "image") 
app.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [8]:
from keras.applications.mobilenet import MobileNet 
from keras.applications.mobilenet import preprocess_input

from keras.applications.mobilenet import decode_predictions 
from PIL import Image 
import numpy as np 
import gradio as gr
model = MobileNet(weights="imagenet", include_top=True)

def resize_image(img, new_w, new_h):
    img = Image.fromarray(img)
    w, h = img.size
    w_scale = new_w / w
    h_scale = new_h / h
    scale = min(w_scale, h_scale)
    resized = img.resize((int(w*scale), int(h*scale)), Image.NEAREST)
    resized = resized.crop((0, 0, new_w, new_h))
    return resized

def predict(input): 
	input_resized = resize_image(input, 224, 224)
	img = np.array(input_resized) 
	img = img.reshape((1, 224, 224, 3))
	img = preprocess_input(img)
	y_pred = model.predict(img, verbose=0)
	label = decode_predictions(y_pred)
	top_prediction = label[0][0]
	formatted_string = "%s (%.2f%%)" % (top_prediction[1], top_prediction[2]*100) 
	return formatted_string
	
app = gr.Interface(fn=predict, inputs=gr.Image(), outputs="text")
app.launch()

2026-05-12 15:54:10.213732: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-12 15:54:10.221689: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-12 15:54:10.604918: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-12 15:54:12.080452: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-12 15:54:13.820284: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warnin

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


In [9]:
from keras.applications.resnet50 import ResNet50
from keras.applications.resnet50 import preprocess_input 
from keras.applications.resnet50 import decode_predictions 
from PIL import Image 
import numpy as np 
import gradio as gr 
model = ResNet50(weights="imagenet", include_top=True)

def resize_image(img, new_w, new_h): 
	img = Image.fromarray(img) 
	w, h = img.size
	w_scale = new_w / w 
	h_scale = new_h / h 
	scale = min(w_scale, h_scale) 
	resized = img.resize((int(w*scale), int(h*scale)), Image.NEAREST) 
	resized = resized.crop((0, 0, new_w, new_h)) 
	return resized

def predict(input):
    input_resized = resize_image(input, 224, 224)
    img = np.array(input_resized)
    img = img.reshape((1, 224, 224, 3)) 
    img = preprocess_input(img)
    y_pred = model.predict(img, verbose=0)
    label = decode_predictions(y_pred)
    max_len = len(label[0]) 
    max_len = 10 if max_len > 10 else max_len
	
    top_10_predictions = {
        label[0][i][1]: float(label[0][i][2])
		for i in range(max_len)
    }

    return top_10_predictions
    
inputs = gr.Image() 
outputs = gr.Label(num_top_classes=3)
app = gr.Interface(
    fn=predict, 
    inputs=inputs, 
    outputs=outputs
) 
app.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


In [10]:
!pip install keras-nlp --upgrade

In [11]:
import keras_nlp
import gradio as gr 

labels = ["負面", "正面"] 
model_name = "bert_tiny_en_uncased_sst2" 
preprocessor = keras_nlp.models.BertPreprocessor.from_preset( model_name, sequence_length=128,)

classifier = keras_nlp.models.BertClassifier.from_preset( model_name, num_classes=2, preprocessor=preprocessor)

def predict(input):
	output = classifier.predict([input])
	predictions = { labels[i]: float(output[0][i]) for i in range(len(labels)) }
	return predictions

outputs = gr.Label(num_top_classes=2) 
examples = ["This movie is good.", "A total waste of my time."] 
app = gr.Interface(fn=predict, inputs="text", outputs=outputs, examples=examples)
app.launch()

100%|████████████████████████████████████████████████████████████████████████| 454/454 [00:00<00:00, 641kB/s]


100%|███████████████████████████████████████████████████████████████████| 1.56k/1.56k [00:00<00:00, 2.55MB/s]


100%|███████████████████████████████████████████████████████████████████| 226k/226k [-00:09<00:00, -24.6kB/s]


100%|███████████████████████████████████████████████████████████████████| 2.83k/2.83k [00:00<00:00, 9.17MB/s]


100%|███████████████████████████████████████████████████████████████████| 50.3M/50.3M [00:06<00:00, 7.85MB/s]


100%|███████████████████████████████████████████████████████████████████| 16.8M/16.8M [00:03<00:00, 4.82MB/s]


* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


2026-05-12 15:56:10.119273: E tensorflow/core/util/util.cc:131] oneDNN supports DT_INT64 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 650ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 294ms/step


In [14]:
from keras_nlp.models import GPT2CausalLMPreprocessor 
from keras_nlp.models import GPT2CausalLM 
import gradio as gr 
import keras 

# 設置混合精度
keras.mixed_precision.set_global_policy("mixed_float16")

preprocessor = GPT2CausalLMPreprocessor.from_preset(
    "gpt2_base_en", 
    sequence_length=128,
)

gpt2_1m = GPT2CausalLM.from_preset(
    "gpt2_base_en", 
    preprocessor=preprocessor
)

def predict(input):
    output = gpt2_1m.generate(input, max_length=200)
    return output

examples = ["My trip to New York was", "My trip to Taipei was"]

app = gr.Interface(
    fn=predict, 
    inputs="text", 
    outputs="text", 
    examples=examples
)

app.launch()


100%|████████████████████████████████████████████████████████████████████████| 431/431 [00:00<00:00, 414kB/s]


100%|████████████████████████████████████████████████████████████████████████| 618/618 [00:00<00:00, 534kB/s]


100%|██████████████████████████████████████████████████████████████████| 0.99M/0.99M [-00:08<00:00, -118kB/s]


100%|██████████████████████████████████████████████████████████████████████| 446k/446k [00:01<00:00, 374kB/s]


100%|█████████████████████████████████████████████████████████████████████| 475M/475M [00:27<00:00, 17.9MB/s]


* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


I0000 00:00:1778572786.285621    3665 service.cc:145] XLA service 0x787c8c00db90 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778572786.285670    3665 service.cc:153]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1778572786.538399    3663 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
